# Fine-tuning: Cardano/Aiken/Hydra Assistant
**Modelo base:** Qwen2.5-Coder-7B-Instruct  
**Método:** QLoRA con unsloth  
**Dataset:** 2069 ejemplos EN/ES de Aiken, Hydra, CIPs  
**Export:** GGUF Q4_K_M para LM Studio  

### Setup previo
1. Runtime → Change runtime type → **A100 GPU**
2. Subir `dataset.jsonl` a Google Drive en `/MyDrive/cardano_ai/dataset.jsonl`

In [ ]:
# Celda 1: Verificar GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# Celda 2: Instalar dependencias
%%capture
!pip install unsloth
!pip install --upgrade transformers datasets trl peft bitsandbytes accelerate

In [ ]:
# Celda 3: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DATASET_PATH = '/content/drive/MyDrive/cardano_ai/dataset.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/cardano_ai/model_output'
GGUF_DIR = '/content/drive/MyDrive/cardano_ai/gguf'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

assert os.path.exists(DATASET_PATH), f'Dataset no encontrado en {DATASET_PATH}'
print(f'Dataset encontrado: {os.path.getsize(DATASET_PATH) / 1024:.1f} KB')

In [ ]:
# Celda 4: Cargar y preparar dataset
import json
from datasets import Dataset

# Leer JSONL
records = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'Total ejemplos: {len(records)}')

from collections import Counter
langs = Counter(r.get('lang', 'unknown') for r in records)
topics = Counter(r.get('topic', 'unknown') for r in records)
print(f'Idiomas: {dict(langs)}')
print(f'Top tópicos: {dict(topics.most_common(8))}')

In [ ]:
# Celda 5: Convertir a formato de chat (ChatML)
# Qwen2.5 usa ChatML: <|im_start|>system ... <|im_end|>

SYSTEM_PROMPT = """You are an expert Cardano blockchain developer specialized in:
- Aiken smart contract language (validators, custom types, stdlib)
- Hydra Head protocol (Layer 2 scaling, off-chain transactions)
- Cardano Improvement Proposals (CIPs)
- eUTxO model (datum, redeemer, script context)
- Plutus and on-chain validation logic

Provide accurate, working code examples. For Aiken code, always use correct syntax with `validator`, `fn`, `when`, `expect` keywords."""

def format_example(record):
    instruction = record.get('instruction', '').strip()
    input_text = record.get('input', '').strip()
    output = record.get('output', '').strip()

    # Combinar instruction + input si hay input
    if input_text:
        user_message = f"{instruction}\n\n{input_text}"
    else:
        user_message = instruction

    # Formato ChatML
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_message}<|im_end|>\n"
        f"<|im_start|>assistant\n{output}<|im_end|>"
    )
    return {'text': text}

# Filtrar ejemplos con campos vacíos
valid_records = [
    r for r in records
    if r.get('instruction') and r.get('output')
    and len(r['output']) > 20
]
print(f'Ejemplos válidos: {len(valid_records)} / {len(records)}')

formatted = [format_example(r) for r in valid_records]
dataset = Dataset.from_list(formatted)

# Split train/eval 95/5
split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split['train']
eval_dataset = split['test']
print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

# Preview
print('\n--- Ejemplo ---')
print(train_dataset[0]['text'][:500])

In [ ]:
# Celda 6: Cargar modelo con unsloth
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048  # suficiente para la mayoría de ejemplos
DTYPE = None           # unsloth detecta automáticamente (bfloat16 en A100)
LOAD_IN_4BIT = True    # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-Coder-7B-Instruct',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print('Modelo cargado.')
print(f'VRAM usada: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# Celda 7: Configurar LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # rank — más alto = más parámetros entrenables
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=32,           # típicamente 2x rank
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # 30% menos VRAM
    random_state=42,
    use_rslora=False,
)

# Ver parámetros entrenables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Parámetros entrenables: {trainable:,} ({100*trainable/total:.2f}% del total)')

In [ ]:
# Celda 8: Configurar entrenamiento
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 16
    warmup_steps=50,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    optim='adamw_8bit',              # menos VRAM que adamw estándar
    weight_decay=0.01,
    seed=42,
    report_to='none',                # cambiar a 'wandb' si querés tracking
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=training_args,
)

print('Trainer configurado.')
print(f'Steps por epoch: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}')
print(f'Total steps: {trainer.args.max_steps if trainer.args.max_steps > 0 else "auto"}')

In [ ]:
# Celda 9: ENTRENAR
# En A100: ~20-30 minutos para 2069 ejemplos x 3 epochs
import time

print('Iniciando entrenamiento...')
start = time.time()

trainer_stats = trainer.train()

elapsed = time.time() - start
print(f'\nEntrenamiento completo en {elapsed/60:.1f} minutos')
print(f'Loss final: {trainer_stats.training_loss:.4f}')

In [ ]:
# Celda 10: Test rápido antes de exportar
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_prompts = [
    'Write a simple Aiken validator that checks if a redeemer equals 42',
    'Escribe un validator de Aiken que verifique que el redeemer es igual a 42',
    'What is the eUTxO model in Cardano?',
]

for prompt in test_prompts:
    inputs = tokenizer(
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n',
        return_tensors='pt'
    ).to('cuda')

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\nQ: {prompt}')
    print(f'A: {response[:400]}')
    print('-' * 60)

In [ ]:
# Celda 11: Guardar adaptadores LoRA (checkpoint ligero)
LORA_DIR = OUTPUT_DIR + '/lora_adapters'
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f'LoRA adapters guardados en: {LORA_DIR}')

In [ ]:
# Celda 12: Exportar a GGUF Q4_K_M para LM Studio
# Esto tarda ~5-10 minutos en A100
print('Exportando a GGUF Q4_K_M...')
print('(Este paso tarda ~5-10 minutos)')

model.save_pretrained_gguf(
    GGUF_DIR + '/cardano-aiken-7b',
    tokenizer,
    quantization_method='q4_k_m',
)

# Listar archivos generados
import os
for f in os.listdir(GGUF_DIR + '/cardano-aiken-7b'):
    size = os.path.getsize(os.path.join(GGUF_DIR + '/cardano-aiken-7b', f))
    print(f'  {f}: {size/1e9:.2f} GB')

print('\nListo para descargar y cargar en LM Studio.')

In [ ]:
# Celda 13 (OPCIONAL): También exportar Q8_0 para mejor calidad si tenés espacio
# model.save_pretrained_gguf(
#     GGUF_DIR + '/cardano-aiken-7b-q8',
#     tokenizer,
#     quantization_method='q8_0',
# )

## Pasos siguientes

1. **Descargar el GGUF** desde Google Drive:
   - Ir a `/MyDrive/cardano_ai/gguf/cardano-aiken-7b/`
   - Descargar el archivo `.gguf` (~4.4 GB)

2. **Cargar en LM Studio:**
   - `My Models` → `Import Model` → seleccionar el `.gguf`
   - GPU layers: máximo posible
   - Context: 8192
   - Temperature: 0.1 para código

3. **System prompt recomendado en LM Studio:**
```
You are an expert Cardano blockchain developer specialized in Aiken smart contracts, 
Hydra Head protocol, and CIP standards. Provide accurate code examples in Aiken syntax.
```